In [2]:
# Password-protected file access (to be used across Jupyter notebooks)

import msoffcrypto # To access password-protected files
import pandas as pd # To read and manipulate data
from io import BytesIO # Temp handling of decrypted files
from getpass import getpass # Get password input w/out encoding

password = getpass("Enter anon file password: ") # Prompt for password
decrypted_file = BytesIO() # Creates temp file in memory

with open(r"C:\Users\karin\Documents\2. Data\Anonymous_Data.xlsx", "rb") as f: # Open the password-protected file
    office_file = msoffcrypto.OfficeFile(f) # Create Officefile object
    office_file.load_key(password=password) # Load password
    office_file.decrypt(decrypted_file) # Decrypt file into memory

df = pd.read_excel(decrypted_file) # Read decrypted file into a DataFrame (DF)
df.shape # Display the shape of the DF (rows and columns)

(2839, 88)

In [3]:
# Rebuild df_binary for attendance feature below

fail_categories = ['Fail Resit', 'Fail Withdraw', 'Repeat without Attendance', 'Repeat with Attendance', 'Complete Repeat'] # Defining categories which count as a fail
df_binary = df[df['Progression Decision'] != 'Trail Progress'].copy() # Create a new DF excluding 'Trail Progress' rows as it is its own edge case
df_binary['initially_failed'] = df_binary['Progression Decision'].isin(fail_categories) # Create new column in new DF: True if student failed and false if they passed

df_binary.shape # Display the shape of the DF (rows and columns)

(2837, 89)

In [4]:
# Attendance Feature: already numeric but accouting for off-site students with null values

df_binary['is_offsite'] = df_binary['Attendance (%)'].isnull() # Create new column which turns true if attendance is null (off-site student)
df_binary['is_offsite'].value_counts() # Count the number of off-site students (True) and on-site students (False)

is_offsite
False    2785
True       52
Name: count, dtype: int64

In [5]:
# Attendance Feature: check if any off-site students are categorised as on-site via admin student status

df_binary['no_attendance_data'] = df_binary['Attendance (%)'].isnull() # Create new column which turns true if attendance is null (off-site student)
df_binary[df_binary['no_attendance_data']]['Student Status'].value_counts() # Count no. students off-site but categorised via admin student status

Student Status
Off-site     41
PR/Repeat    10
Normal        1
Name: count, dtype: int64

In [6]:
# Breakdown of student numbers via three student statuses

df_binary['is_offsite'] = df_binary['Student Status'].str.contains('Off-site', case=False, na=False) # Create new column which turns true if student status contains 'Off-site' (off-site student)
df_binary['is_repeating'] = df_binary['Student Status'].str.contains('PR/Repeat', case=False, na=False) # Create new column which turns true if student status contains 'PR/Repeat' (repeating student)
df_binary['unexplained_null_attendance'] = df_binary['no_attendance_data'] & ~df_binary['is_offsite'] & ~df_binary['is_repeating'] # Create new column which turns true if student has no attendance data but is not off-site or repeating

df_binary[['is_offsite', 'is_repeating', 'unexplained_null_attendance']].sum() # Count students in each category

is_offsite                     42
is_repeating                   92
unexplained_null_attendance     1
dtype: int64

In [7]:
# Dividing repeating students into two categories: those with and without attendance expectation

df_binary['is_repeating'] = df_binary['is_repeating'] # All repeating students for general repeat flag
df_binary['repeating_with_no_attendance_expectation'] = df_binary['is_repeating'] & df_binary['Attendance (%)'].isnull() # Those repeating without attendance expectation
df_binary['repeating_with_attendance'] = df_binary['is_repeating'] & df_binary['Attendance (%)'].notnull() # Those repeating with attendance expectation

df_binary[['repeating_with_no_attendance_expectation', 'repeating_with_attendance']].sum() # Count students in each category

repeating_with_no_attendance_expectation    10
repeating_with_attendance                   82
dtype: int64

In [8]:
# Rebuild list of VLE columns

vle_columns = [col for col in df.columns if 'VLE' in col] # List VLE columns
len(vle_columns) # Display the number of VLE columns

19

In [9]:
# VLE Engagement Score

rag_map = {'RED': 0, 'AMBER': 1, 'GREEN': 2} # Ordinal mapping for RAG values, the higher, the better the engagement score

vle_numeric = df_binary[vle_columns].apply(lambda col: col.str.upper()) # Convert all RAG values to uppercase to avoid errors
vle_numeric = vle_numeric.replace(rag_map).replace('GREY', pd.NA) # Replace RAG values with numeric values and replace GREY with null
vle_numeric = vle_numeric.apply(pd.to_numeric, errors='coerce') # Convert all values to numeric

df_binary['vle_avg_score'] = vle_numeric.mean(axis=1) # Calculate the average VLE score for each student across all weeksand store in a new column
df_binary['vle_red_weeks'] = (vle_numeric == 0).sum(axis=1) # Count the number of RED weeks for each student
df_binary['vle_grey_weeks'] = vle_numeric.isnull().sum(axis=1) # Count the number of GREY weeks for each student
df_binary['has_grey_vle'] = df_binary['vle_grey_weeks'] > 3 # Binary flag check if student has more than 3 GREY weeks (adjusted as fewer likely reflects late registration)

df_binary[['vle_avg_score', 'vle_red_weeks', 'vle_grey_weeks', 'has_grey_vle']].describe() # Print the scores

,vle_avg_score,vle_red_weeks,vle_grey_weeks
count,2746.000000,2837.000000,2837.000000
mean,1.315146,3.348255,1.381036
std,0.499003,4.320137,4.126316
min,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000
50%,1.421053,2.000000,0.000000
75%,1.736842,5.000000,0.000000
max,2.000000,19.000000,19.000000


In [10]:
# Flag for no VLE data students

df_binary['no_vle_data'] = df_binary['vle_avg_score'].isnull() # Pulls 91 students who's overall VLE engagement is null (grey or non-existant)
df_binary['no_vle_data'].sum() # Check for 91 returned

91

In [11]:
# Encoding categoricals

categorical_cols = ['Department', 'Student Status', 'Year Group'] # Columns that need to be one-hot encoded - Added Year Group
df_encoded = pd.get_dummies(df_binary, columns=categorical_cols, drop_first=False) # New column created per category as True or False
df_encoded = df_encoded.drop(columns=['Admin Status']) # Drop Admin Status due to data leakage
df_encoded.shape # Check new column count

(2837, 108)

In [12]:
# Run check on columns to confirm

[col for col in df_encoded.columns if 'Admin Status' in col or 'Student Status' in col or 'Department' in col]

['Department_Department A',
 'Department_Department B',
 'Department_Department C',
 'Student Status_Normal',
 'Student Status_Off-site',
 'Student Status_PR/Repeat']

In [13]:
# Final check on rows and columns

df_encoded.shape # Should remain as 2837, 110 (no rows lost or duplicated during the encoding)

(2837, 108)

In [14]:
# Dropping columns used for manual checks

df_encoded = df_encoded.drop(columns=['Notes', 'check'])
df_encoded.shape # check the two columns are dropped

(2837, 106)

In [15]:
# Final check on nulls (or greys)

null_check = df_encoded.isnull().sum() # Count any nulls per column
null_check[null_check > 0] # Show only the columns that have nulls

Attendance (%)                       52
Week 1 & 2 Attendance %             129
Week 2 & 3 Attendance %             106
Week 3 & 4 Attendance %              90
Week 4 & 5 Attendance %              86
Week 5 & 6 Attendance %              86
Week 6 & 7 Attendance %              84
Week 7 & 8 Attendance %              82
Week 8 & 9 Attendance %              81
Week 9 & 10 Attendance %             81
Week 11 & 12 Attendance %           110
Week 11 & 12 Attendance RAG          21
Week 11 & 12 VLE Engagement RAG      21
Week 11 & 12 COMBINED RAG            21
Week 12 & 13 Attendance %           110
Week 12 & 13 Attendance RAG          21
Week 12 & 13 VLE Engagement RAG      21
Week 12 & 13 COMBINED RAG            21
Week 13 & 14 Attendance %           110
Week 13 & 14 Attendance RAG          21
Week 13 & 14 VLE Engagement RAG      21
Week 13 & 14 COMBINED RAG            21
Week 14 & 15 Attendance %           110
Week 14 & 15 Attendance RAG          21
Week 14 & 15 VLE Engagement RAG      21


Cells above copied over from Notebook 8.

In [17]:
# X/y split

leakage_and_id_cols = ['Academic Year', 'Student_ID', 'Progression Decision', 'Resit?', 'Progression Decision (Resit)', 'is_repeating', 'repeating_with_attendance', 'repeating_with_no_attendance_expectation'] # Columns which will be excluded as information is not relevant to prediction
raw_rag_cols = [col for col in df_encoded.columns if 'RAG' in col and 'has_grey' not in col] # Gets all raw weekly RAG text columns as there is summary features
weekly_pct_cols = [col for col in df_encoded.columns if '&' in col and 'Attendance %' in col] # Gets all raw weekly attendance percentage columns as there is overall attendance percentage

cols_to_drop = leakage_and_id_cols + raw_rag_cols + weekly_pct_cols # Get all columns into one place to remove
X = df_encoded.drop(columns=cols_to_drop + ['initially_failed']) # Create new table but remove said columns as well as the true outcome (as that is the target)
y = df_encoded['initially_failed'] # Single target column

X.shape, y.shape # Print shapes

((2837, 21), (2837,))

In [18]:
# Student Status counts of students with null

excluded_mask = X[['Attendance (%)', 'vle_avg_score']].isnull().any(axis=1) 

X_excluded_check = df_encoded[excluded_mask] # Pull all rows for the 103 students
X_excluded_check[['Student Status_Off-site', 'Student Status_PR/Repeat', 'Student Status_Normal']].sum() # Count how many students fall into each category

Student Status_Off-site     42
Student Status_PR/Repeat    60
Student Status_Normal        1
dtype: int64

In [19]:
# Implement null handing (exclude from data)

X_excluded = X[excluded_mask].copy() # Pull 103 students features into a seperate table
y_excluded = y[excluded_mask].copy() # Pull 103 outcomes for the students

X_modelling = X[~excluded_mask].copy() # Everyone else BUT the 103 students
y_modelling = y[~excluded_mask].copy() # Same as above for outcomes

X_modelling.shape, X_excluded.shape

((2734, 21), (103, 21))

In [20]:
# New Train & Test Split post exclusion - same steps as before just change of data

from sklearn.model_selection import train_test_split # Import the split function

X_train, X_test, y_train, y_test = train_test_split(
    X_modelling, y_modelling,
    test_size=0.2,
    stratify=y_modelling,
    random_state=22
)
X_train.shape, X_test.shape

((2187, 21), (547, 21))

In [ ]:
# Attendance only-feature subset - same train/test students as the multi-signal models but single feature only for comparison

X_train_attendance = X_train[['Attendance (%)']]
X_test_attendance = X_test[['Attendance (%)']]

X_train_attendance.shape, X_test_attendance.shape

((2187, 1), (547, 1))

In [28]:
# SMOTE application

from imblearn.over_sampling import SMOTE # Importing SMOTE for oversampling minority class

attendance_smote = SMOTE(random_state=22)
X_train_attendance_smote, y_train_attendance_smote = attendance_smote.fit_resample(X_train_attendance, y_train) # Resamples the training data only leaving test untouched

y_train_attendance_smote.value_counts() 

initially_failed
False    1377
True     1377
Name: count, dtype: int64

In [29]:
# Refit Random Forest on single signal (attendance-only) data using same tuned hyperparameters as multi-signal Random Forest Tuned

from sklearn.ensemble import RandomForestClassifier

rf_attendance_tuned = RandomForestClassifier(
    n_estimators=300,
    max_depth=30,
    min_samples_split=10,
    min_samples_leaf=1,
    random_state=22
)

rf_attendance_tuned.fit(X_train_attendance_smote, y_train_attendance_smote) # Fit on attendance-only data

y_pred_rf_attendance = rf_attendance_tuned.predict(X_test_attendance) # Predict on untouched attendance-only test set

# Evaluate on untouched test set

from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred_rf_attendance)) # Pecision, recall, f1-score for tuned model

y_fail_proba_rf_attendance = rf_attendance_tuned.predict_proba(X_test_attendance)[:,1] # Probabilities for AUC-ROC
print('AUC-ROC:', roc_auc_score(y_test, y_fail_proba_rf_attendance))

              precision    recall  f1-score   support

       False       0.77      0.67      0.72       344
        True       0.54      0.66      0.60       203

    accuracy                           0.67       547
   macro avg       0.66      0.67      0.66       547
weighted avg       0.69      0.67      0.67       547

AUC-ROC: 0.7341476686905717


The above shows that a single signal (attendance-only) only slightly beats the multi-signal on recall (from 0.63 to 0.66). However, this happens at a real cost in other areas. Precision has had a large drop (from 0.64 to 0.54) meaning more false alarms so it flags students who are not at risk. In addition, AUC-ROC drops meaningfully from 0.7931 to 0.7341. F1 dropping too from 0.64 to 0.60 reveals the balance shifts negatively showing the drop in precision outweighs the small recall gain.